In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize_scalar

In [2]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, LabelEncoder
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

In [3]:
df = pd.read_csv('retail_price.csv')
df.head()

,product_id,product_category_name,month_year,qty,total_price,freight_price,unit_price,product_name_lenght,product_description_lenght,product_photos_qty,...,comp_1,ps1,fp1,comp_2,ps2,fp2,comp_3,ps3,fp3,lag_price
0,bed1,bed_bath_table,01-05-2017,1,45.95,15.100000,45.95,39,161,2,...,89.9,3.9,15.011897,215.000000,4.4,8.760000,45.95,4.0,15.100000,45.90
1,bed1,bed_bath_table,01-06-2017,3,137.85,12.933333,45.95,39,161,2,...,89.9,3.9,14.769216,209.000000,4.4,21.322000,45.95,4.0,12.933333,45.95
2,bed1,bed_bath_table,01-07-2017,6,275.70,14.840000,45.95,39,161,2,...,89.9,3.9,13.993833,205.000000,4.4,22.195932,45.95,4.0,14.840000,45.95
3,bed1,bed_bath_table,01-08-2017,4,183.80,14.287500,45.95,39,161,2,...,89.9,3.9,14.656757,199.509804,4.4,19.412885,45.95,4.0,14.287500,45.95
4,bed1,bed_bath_table,01-09-2017,2,91.90,15.100000,45.95,39,161,2,...,89.9,3.9,18.776522,163.398710,4.4,24.324687,45.95,4.0,15.100000,45.95


In [4]:
# derived competitive features
df["avg_comp_price"]   = (df["comp_1"] + df["comp_2"] + df["comp_3"]) / 3
df["price_vs_comp"]    = df["unit_price"] / df["avg_comp_price"]          # >1 = premium, <1 = discount
df["price_gap_comp1"]  = df["unit_price"] - df["comp_1"]
df["log_unit_price"]   = np.log(df["unit_price"])
df["log_qty"]          = np.log(df["qty"])
df["revenue"]          = df["unit_price"] * df["qty"]
df["margin_proxy"]     = df["unit_price"] - df["freight_price"]
 
print("=" * 60)
print("  DATASET OVERVIEW")
print("=" * 60)
print(f"  Rows         : {len(df)}")
print(f"  Products     : {df['product_id'].nunique()}")
print(f"  Categories   : {df['product_category_name'].nunique()}")
print(f"  Date range   : {df['month_year'].min()} → {df['month_year'].max()}")
print(f"  Price range  : R${df['unit_price'].min():.2f} – R${df['unit_price'].max():.2f}")
print(f"  Demand range : {df['qty'].min()} – {df['qty'].max()} units/month")
print()

  DATASET OVERVIEW
  Rows         : 676
  Products     : 52
  Categories   : 9
  Date range   : 01-01-2017 → 01-12-2017
  Price range  : R$19.90 – R$364.00
  Demand range : 1 – 122 units/month



### 1.  PRICE ELASTICITY OF DEMAND  (Log-Log OLS)

In [5]:
print("=" * 60)
print("  MODULE 1 — PRICE ELASTICITY (Log-Log OLS)")
print("=" * 60)
 
X_elas = df[["log_unit_price"]].copy()
X_elas = np.column_stack([np.ones(len(df)), X_elas])   # add intercept
y_elas = df["log_qty"].values
 
# OLS via numpy (exact, for interpretability)
beta, res, rank, sv = np.linalg.lstsq(X_elas, y_elas, rcond=None)
intercept_elas, elasticity = beta
 
# standard errors
n, k = X_elas.shape
y_pred_elas = X_elas @ beta
ss_res = np.sum((y_elas - y_pred_elas) ** 2)
ss_tot = np.sum((y_elas - y_elas.mean()) ** 2)
r2_elas = 1 - ss_res / ss_tot
mse_elas = ss_res / (n - k)
var_beta = mse_elas * np.linalg.inv(X_elas.T @ X_elas)
se_beta  = np.sqrt(np.diag(var_beta))
t_stats  = beta / se_beta
p_vals   = 2 * (1 - stats.t.cdf(np.abs(t_stats), df=n-k))
 
print(f"\n  Log-Log Regression:  log(qty) = α + ε·log(price)")
print(f"  ─────────────────────────────────────────────────")
print(f"  Intercept  (α)  : {intercept_elas:+.4f}   SE={se_beta[0]:.4f}  p={p_vals[0]:.4f}")
print(f"  Elasticity (ε)  : {elasticity:+.4f}   SE={se_beta[1]:.4f}  p={p_vals[1]:.4f}")
print(f"  R²              : {r2_elas:.4f}")
print()
print(f"  INTERPRETATION:")
print(f"  → A 1% price increase → {elasticity:.2f}% change in demand")
if elasticity < -1:
    print(f"  → Demand is ELASTIC (|ε|>1): customers are price-sensitive")
elif -1 <= elasticity < 0:
    print(f"  → Demand is INELASTIC (|ε|<1): customers less price-sensitive")
else:
    print(f"  → Positive/zero elasticity — price may act as quality signal")
 
# Per-category elasticities
print(f"\n  PER-CATEGORY ELASTICITIES:")
print(f"  {'Category':<30} {'Elasticity':>12} {'R²':>8} {'N':>5}")
print(f"  {'-'*57}")
cat_results = []
for cat, grp in df.groupby("product_category_name"):
    if len(grp) < 5:
        continue
    xc = np.column_stack([np.ones(len(grp)), np.log(grp["unit_price"])])
    yc = np.log(grp["qty"].values)
    bc, _, _, _ = np.linalg.lstsq(xc, yc, rcond=None)
    yp = xc @ bc
    r2c = 1 - np.sum((yc - yp)**2) / np.sum((yc - yc.mean())**2)
    cat_results.append({"category": cat, "elasticity": bc[1], "r2": r2c, "n": len(grp)})
    print(f"  {cat:<30} {bc[1]:>+12.3f} {r2c:>8.3f} {len(grp):>5}")
print()

  MODULE 1 — PRICE ELASTICITY (Log-Log OLS)

  Log-Log Regression:  log(qty) = α + ε·log(price)
  ─────────────────────────────────────────────────
  Intercept  (α)  : +2.7681   SE=0.2807  p=0.0000
  Elasticity (ε)  : -0.1313   SE=0.0623  p=0.0355
  R²              : 0.0065

  INTERPRETATION:
  → A 1% price increase → -0.13% change in demand
  → Demand is INELASTIC (|ε|<1): customers less price-sensitive

  PER-CATEGORY ELASTICITIES:
  Category                         Elasticity       R²     N
  ---------------------------------------------------------
  bed_bath_table                       +0.574    0.066    61
  computers_accessories                +0.398    0.009    69
  consoles_games                       -2.276    0.274    22
  cool_stuff                           +0.137    0.003    57
  furniture_decor                      +0.308    0.012    48
  garden_tools                         -0.799    0.086   160
  health_beauty                        -0.144    0.023   130
  perfumery   

# ─────────────────────────────────────────────────────────
### 2.  SIMPLE PRICE OPTIMIZATION (Revenue Maximization)
# ─────────────────────────────────────────────────────────

In [6]:
print("=" * 60)
print("  MODULE 2 — SIMPLE REVENUE-OPTIMAL PRICE")
print("=" * 60)
print("""
  Revenue = Price × Quantity
  Using elasticity model: Q(p) = exp(α) × p^ε
  → Revenue(p) = p × exp(α) × p^ε = exp(α) × p^(1+ε)
  → dR/dp = 0  →  p* = 0 (if ε<-1, revenue maximized at 0)
  → Real world: Lerner condition  p* = MC / (1 + 1/ε)
""")
 
# Lerner markup rule for each category
MC_proxy = df.groupby("product_category_name")["freight_price"].mean()
print(f"  {'Category':<30} {'Elasticity':>12} {'MC Proxy':>10} {'Optimal Price':>14} {'Avg Actual':>12}")
print(f"  {'-'*80}")
for r in cat_results:
    cat = r["category"]
    e   = r["elasticity"]
    mc  = MC_proxy.get(cat, 15)
    avg_p = df[df["product_category_name"] == cat]["unit_price"].mean()
    if e < -1:
        p_star = mc / (1 + 1/e)
        flag = ""
    elif e < 0:
        p_star = avg_p  # inelastic — pricing power, no closed form without cost
        flag = "*"
    else:
        p_star = avg_p
        flag = "†"
    print(f"  {cat:<30} {e:>+12.3f} {mc:>10.2f} {p_star:>14.2f}{flag} {avg_p:>12.2f}")
 
print(f"\n  * Inelastic: increase price (pricing power region)")
print(f"  † Non-negative elasticity: investigate data quality / other drivers")


  MODULE 2 — SIMPLE REVENUE-OPTIMAL PRICE

  Revenue = Price × Quantity
  Using elasticity model: Q(p) = exp(α) × p^ε
  → Revenue(p) = p × exp(α) × p^ε = exp(α) × p^(1+ε)
  → dR/dp = 0  →  p* = 0 (if ε<-1, revenue maximized at 0)
  → Real world: Lerner condition  p* = MC / (1 + 1/ε)

  Category                         Elasticity   MC Proxy  Optimal Price   Avg Actual
  --------------------------------------------------------------------------------
  bed_bath_table                       +0.574      16.14          78.63†        78.63
  computers_accessories                +0.398      25.10         119.48†       119.48
  consoles_games                       -2.276      14.81          26.42        27.03
  cool_stuff                           +0.137      18.98         107.86†       107.86
  furniture_decor                      +0.308      16.94          60.15†        60.15
  garden_tools                         -0.799      28.46          80.09*        80.09
  health_beauty                 